In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
from skimage.io import imread, imsave
from skimage.feature import multiscale_basic_features
from skimage.morphology import opening, closing
from scipy.ndimage import label, gaussian_gradient_magnitude
import onnxruntime as rt

from snap_to_edge import snap_labels_to_edge
from calmutils.morphology.structuring_elements import hypersphere_centered

In [ ]:
in_path = '/Volumes/nn/Julia Vogtmann/Microscopy/25JV_006'
model_info_path = '/Users/david/Desktop/jurkat_nucleolin/rf_nucleus_nucleolin_002_info.json'

# subdirectory containing images to segment (in TIFF format)
image_subdirectory = 'tif'
# pattern of files to segment
image_file_pattern = '*_ch1.tif' # e.g. "*_ch0.tif" to only include images of one channel 
# output directory to save resulting masks to
out_subdirectory = 'segmentation_nucleoli'

# we will do morphological opening (removes isolated foreground pixels)
# followed by closing (removes small holes)
# set radii to 0 to skip
opening_radius = 2
closing_radius = 2

# whether to do connected-component labelling
do_labelling = False

# whether to do snap-to-edge of Gaussian gradient magnitude of image
do_snap_to_edge = True
snap_to_edge_radius = 2
snap_to_edge_ggm_sigma = 1.0

In [ ]:
# load model info
with open(model_info_path) as f:
    info_dict = json.load(f)

# arguments to multiscale_basic_features
feature_fun_kwargs = info_dict['feature_fun_kwargs']

# get model path (NOTE: needs to be in same folder as info file)
model_path = Path(model_info_path).parent / info_dict['model_file']

# init ONNX session with model
sess = rt.InferenceSession(model_path, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name

In [ ]:
# get all input files and show for verification
in_files = sorted((Path(in_path) / image_subdirectory).glob(image_file_pattern))
in_files

In [ ]:
info_dict

In [ ]:
# make output directory if it does not yet exist
out_dir = Path(in_path) / out_subdirectory
if not out_dir.exists():
    out_dir.mkdir()

for in_file in in_files:

    # load image file and extract features
    img = imread(in_file).astype(float)

    print(f'loaded {in_file}.')
    
    # apply plane-by-plane if model is 2d but data is 3d
    if (img.ndim != 2) and info_dict['is_2d']:
        features = np.stack([multiscale_basic_features(img_i, **feature_fun_kwargs) for img_i in img])
    else:
        features = multiscale_basic_features(img, **feature_fun_kwargs)

    print(f'extracted features for {in_file}.')
    
    features_flat = features.reshape((-1, features.shape[-1]))

    # predict pixel-wise classification, reshape to image shape
    pred_onx = sess.run([label_name], {input_name: features_flat.astype(np.float32)})[0]
    mask_pred = pred_onx.reshape(img.shape)

    print(f'predicted {in_file}.')

    # do morphological opening & closing
    if opening_radius > 0:
        mask_pred = opening(mask_pred, hypersphere_centered(img.ndim, opening_radius))
    if closing_radius > 0:
        mask_pred = closing(mask_pred, hypersphere_centered(img.ndim, closing_radius))

    # connected-components labelling (instance seg.)
    if do_labelling:
        mask_pred, _ = label(mask_pred)

    # snap to edge
    if do_snap_to_edge:
        mask_pred = snap_labels_to_edge(mask_pred, gaussian_gradient_magnitude(img, snap_to_edge_ggm_sigma), snap_to_edge_radius)

    # generate output file name
    out_file = out_dir / (in_file.stem + '_segmented.tif')

    # save, ignore warnings about low contrast
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        imsave(out_file, mask_pred.astype(np.uint16))

    print(f'segmented {in_file}.')